# MoE-MedIR — Kaggle Fine-Tune Notebook

> **Paper**: Bridging the Intra-Modal Heterogeneity Gap: Mixture-of-Experts for Multi-Modality Medical Image Retrieval
> **Conference**: ICARCV 2026

Fine-tune backbone + MoE head end-to-end trên raw images (không dùng pre-extracted features).

**Chiến lược 2 Stage:**
- Stage 1 (`frozen_epochs` đầu): backbone frozen, chỉ train MoE head (warm-up)
- Stage 2 (còn lại): unfreeze last N blocks (ViT) / full model (CNN), backbone_lr nhỏ hơn

**CNN** (convnext_base): unfreeze toàn bộ model ở Stage 2
**ViT** (clip, biomedclip, dino): chỉ unfreeze `finetune_blocks` blocks cuối + final norm

**Setup**: Notebook → Settings → Accelerator → GPU T4/P100
**Internet**: Settings → Internet → **On**

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Settings → Accelerator → GPU')

## Cell 2 — Install Packages

In [ ]:
%%capture
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'medmnist', 'open_clip_torch', 'timm', 'transformers',
    'pytorch-metric-learning', 'scikit-learn',
    'matplotlib', 'seaborn'], check=True)
print('Done.')

## Cell 3 — Clone Project từ GitHub

**Kaggle**: Settings → Internet phải **On**.

In [ ]:
import os, subprocess

REPO_URL     = 'https://github.com/trong5nhan6/moe_medir.git'
PROJECT_ROOT = '/kaggle/working/moe_medir'

if os.path.exists(PROJECT_ROOT):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'])

## Cell 4 — ⚙️ Config Editor

**Chỉnh sửa tại đây** trước khi chạy fine-tune.

| Tham số | Các lựa chọn |
|---|---|
| `BACKBONE` | `clip_vitb32` · `biomedclip` · `dinov2_vitb14` · `convnext_base` |
| `FEATURE_MODE` | `concat` (CLS+Patch / GAP+GMP, 2×dim) · `cls` (CLS / GAP only, dim) |
| `ROUTING_MODE` | `token_choice` · `expert_choice` |
| `TOP_K` | 2 (sparse) · 4 (dense) |

**Fine-tuning hyperparams:**

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `FROZEN_EPOCHS` | 5 | Stage 1: warm-up với backbone frozen |
| `FINETUNE_BLOCKS` | 2 | Số blocks cuối unfreeze (ViT) |
| `BACKBONE_LR` | 1e-5 | LR cho backbone (Stage 2) |
| `HEAD_LR` | 1e-4 | LR cho MoE head (cả 2 stage) |

In [ ]:
import re, sys, importlib

# ════════════════════════════════════════════════════════════════
#  ✏️  CHỈNH SỬA CONFIG TẠI ĐÂY
# ════════════════════════════════════════════════════════════════

BACKBONE      = "clip_vitb32"   # clip_vitb32 | biomedclip | dinov2_vitb14 | convnext_base
FEATURE_MODE  = "concat"        # concat | cls
ROUTING_MODE  = "token_choice"  # token_choice | expert_choice
TOP_K         = 2
EMBED_DIM     = 128
NUM_EXPERTS   = 8
BATCH_SIZE    = 64              # nhỏ hơn vì images nặng hơn features
EPOCHS        = 50

# Fine-tuning specific
FROZEN_EPOCHS   = 5     # Stage 1: frozen warm-up epochs
FINETUNE_BLOCKS = 2     # ViT: số blocks cuối unfreeze; CNN: bỏ qua (full model)
BACKBONE_LR     = 1e-5  # LR cho backbone (Stage 2)
HEAD_LR         = 1e-4  # LR cho MoE head

# ════════════════════════════════════════════════════════════════

def _patch(content, field, value):
    if isinstance(value, str):
        pattern     = rf'([ \t]+{field}\s*:\s*\w+\s*=\s*)["\'\'].*?["\'\']'
        replacement = rf'\g<1>"{value}"'
    else:
        pattern     = rf'([ \t]+{field}\s*:\s*[\w\[\]]+\s*=\s*)[\d.e+\-]+'
        replacement = rf'\g<1>{value}'
    new, n = re.subn(pattern, replacement, content)
    if n == 0:
        print(f"  [warn] field '{field}' not found")
    return new

with open('config.py', 'r') as f:
    cfg_text = f.read()

for field, val in [
    ('backbone',     BACKBONE),
    ('feature_mode', FEATURE_MODE),
    ('routing_mode', ROUTING_MODE),
    ('top_k',        TOP_K),
    ('embed_dim',    EMBED_DIM),
    ('num_experts',  NUM_EXPERTS),
    ('batch_size',   BATCH_SIZE),
    ('epochs',       EPOCHS),
    ('lr',           HEAD_LR),
]:
    cfg_text = _patch(cfg_text, field, val)

with open('config.py', 'w') as f:
    f.write(cfg_text)

if 'config' in sys.modules:
    import config as _c; importlib.reload(_c); from config import CFG
else:
    from config import CFG

print("=" * 60)
print("  Config hiện tại")
print("=" * 60)
print(f"  backbone      : {CFG.backbone}")
print(f"  feature_mode  : {CFG.feature_mode}  →  feature_dim={CFG.feature_dim}")
print(f"  routing_mode  : {CFG.routing_mode}")
print(f"  top_k         : {CFG.top_k}  /  num_experts={CFG.num_experts}")
print(f"  embed_dim     : {CFG.embed_dim}")
print(f"  batch_size    : {CFG.batch_size}  epochs={CFG.epochs}")
print("=" * 60)
print(f"  frozen_epochs  : {FROZEN_EPOCHS}  (Stage 1 warm-up)")
print(f"  finetune_blocks: {FINETUNE_BLOCKS}  blocks unfreeze (ViT only)")
print(f"  backbone_lr    : {BACKBONE_LR}  (Stage 2)")
print(f"  head_lr        : {HEAD_LR}")
print("=" * 60)
print("✅  config.py đã được cập nhật.")

## Cell 5 — Fine-Tune: 2-Stage Training

Chạy `train_finetune.py` với cấu hình từ Cell 4.

**Tiến trình:**
- **Stage 1** (epochs 1–`FROZEN_EPOCHS`): Backbone frozen → chỉ MoE head học
- **Stage 2** (epochs `FROZEN_EPOCHS+1`–`EPOCHS`): Unfreeze → cả backbone + head học

Log hiển thị `[S1]` / `[S2]` mỗi batch. Val mAP@R được đánh giá mỗi 5 epochs.

~25–40 phút trên P100.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, 'train_finetune.py',
    '--model', 'moe',
    '--epochs',          str(EPOCHS),
    '--frozen_epochs',   str(FROZEN_EPOCHS),
    '--finetune_blocks', str(FINETUNE_BLOCKS),
    '--backbone_lr',     str(BACKBONE_LR),
    '--head_lr',         str(HEAD_LR),
], check=True)

## Cell 6 — Evaluate Fine-tuned Model

In [ ]:
import subprocess, sys
from config import CFG
run_name = f'finetune_{CFG.backbone}_moe'
subprocess.run([sys.executable, 'eval/evaluate.py',
                '--model', 'moe',
                '--run_name', run_name], check=True)

## Cell 7 — Training History & Best Score

In [ ]:
import pandas as pd, os
from config import CFG

run_name  = f'finetune_{CFG.backbone}_moe'
hist_path = f'results/history_{run_name}.csv'

if os.path.exists(hist_path):
    df = pd.read_csv(hist_path)
    print('=== Training History ===')
    cols = ['epoch', 'stage', 'avg_mAP@R', 'loss', 'sc_loss', 'orth_loss']
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    print()
    best_row = df.loc[df['avg_mAP@R'].idxmax()]
    print(f"Best val mAP@R: {best_row['avg_mAP@R']:.2f}  (epoch {int(best_row['epoch'])}, Stage {int(best_row['stage'])})")
else:
    print(f'File chưa có: {hist_path}')

## Cell 8 — Save Results

Tất cả file trong `/kaggle/working/` tự động lưu — tải về qua tab **Output**.

In [ ]:
import os
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        path = os.path.join(root, f)
        print(f'{path}  ({os.path.getsize(path)/1024:.1f} KB)')